# LAB 1: Instalação e Configuração do Docling
## Aula 5 — Docling e Ingestão Inteligente de Documentos
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração estimada:** 45 minutos  
**📚 Pré-requisitos:** Aulas 1–4 concluídas (RAG, chunking, embeddings, OpenSearch)  
**🎯 Objetivos deste laboratório:**
- Instalar e configurar o Docling 2.x no Google Colab
- Converter um documento PDF jurídico para Markdown estruturado
- Explorar o modelo de documento DoclingDocument
- Comparar a qualidade da extração Docling vs. extração simples (pdfplumber)
- Gerar os primeiros chunks com metadados jurídicos

## 📋 Pré-requisitos e Ambiente

**O que vamos usar neste lab:**
- `docling` — motor de extração de documentos da IBM Research
- `pdfplumber` — extração simples de PDF (para comparação)
- `reportlab` — geração de PDFs de exemplo para teste
- `sentence-transformers` — modelo BGE-M3 para embeddings

**Atenção:** A instalação do Docling pode demorar 3–5 minutos no Colab (inclui modelos de ML).


In [ ]:
# Instalação das dependências
!pip install docling>=2.0.0 --quiet
!pip install pdfplumber reportlab sentence-transformers --quiet
print("✅ Instalação concluída!")

In [ ]:
import sys, warnings, os, json, re
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

assert sys.version_info >= (3, 11), f'Requer Python 3.11+'
print(f"✅ Python {sys.version.split()[0]}")

import docling, pdfplumber, reportlab
print("✅ Todas as dependências verificadas com sucesso!")
print(f"   • pdfplumber: {pdfplumber.__version__}")

## 📄 Etapa 1: Criar Documento PDF Jurídico de Exemplo

Como estamos no Colab sem acesso a documentos reais, vamos criar um PDF de acórdão
fictício com estrutura realista usando `reportlab`. Este PDF terá:
- Cabeçalho do tribunal
- Ementa estruturada
- Corpo do acórdão com múltiplas seções

**⏱️ Tempo estimado: 2 minutos**


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib import colors

CAMINHO_PDF = '/tmp/acordao_stj_exemplo.pdf'

doc = SimpleDocTemplate(CAMINHO_PDF, pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)

styles = getSampleStyleSheet()
estilo_titulo = ParagraphStyle('titulo', parent=styles['Heading1'],
    fontSize=14, alignment=TA_CENTER, spaceAfter=12)
estilo_negrito = ParagraphStyle('negrito', parent=styles['Normal'],
    fontSize=10, fontName='Helvetica-Bold', spaceAfter=6)
estilo_corpo = ParagraphStyle('corpo', parent=styles['Normal'],
    fontSize=10, alignment=TA_JUSTIFY, spaceAfter=8, leading=14)
estilo_ementa = ParagraphStyle('ementa', parent=styles['Normal'],
    fontSize=9, alignment=TA_JUSTIFY, leftIndent=20, rightIndent=20, spaceAfter=8, leading=13)

elementos = []
elementos.append(Paragraph('SUPERIOR TRIBUNAL DE JUSTIÇA', estilo_titulo))
elementos.append(Paragraph('SEXTA TURMA', estilo_titulo))
elementos.append(HRFlowable(width='100%', thickness=1, color=colors.black))
elementos.append(Spacer(1, 0.3*cm))
elementos.append(Paragraph('<b>HABEAS CORPUS Nº 892.341-SP (2024/0089234-1)</b>', estilo_negrito))
elementos.append(Paragraph('<b>RELATOR:</b> MIN. ROBERTO ALVES DA SILVA', estilo_corpo))
elementos.append(Paragraph('<b>IMPETRANTE:</b> DEFENSORIA PÚBLICA DO ESTADO DE SÃO PAULO', estilo_corpo))
elementos.append(Paragraph('<b>PACIENTE:</b> CARLOS EDUARDO MENDONÇA', estilo_corpo))
elementos.append(Spacer(1, 0.5*cm))

elementos.append(Paragraph('<b>EMENTA</b>', estilo_negrito))
elementos.append(Paragraph(
    'HABEAS CORPUS. TRÁFICO DE ENTORPECENTES (ART. 33 DA LEI N. 11.343/2006). '
    'PRISÃO PREVENTIVA. FUNDAMENTAÇÃO INIDÔNEA. AUSÊNCIA DE ELEMENTOS CONCRETOS '
    'INDIVIDUALIZADOS. GRAVIDADE ABSTRATA DO DELITO. INSUFICIÊNCIA. '
    'CONSTRANGIMENTO ILEGAL CONFIGURADO. ORDEM CONCEDIDA.', estilo_ementa))
elementos.append(Paragraph(
    '1. A prisão preventiva, medida de caráter excepcional e subsidiário, exige fundamentação '
    'idônea, calcada em dados concretos e individualizados que demonstrem a necessidade '
    'da custódia cautelar para garantia da ordem pública (art. 312 do CPP).', estilo_ementa))
elementos.append(Paragraph(
    '2. A simples referência à gravidade abstrata do delito, sem indicação de elementos '
    'concretos que demonstrem periculosidade do agente, não constitui fundamentação '
    'idônea para a prisão preventiva (Súmula 440/STJ).', estilo_ementa))
elementos.append(HRFlowable(width='100%', thickness=0.5, color=colors.grey))
elementos.append(Spacer(1, 0.5*cm))

elementos.append(Paragraph('<b>RELATÓRIO</b>', estilo_negrito))
elementos.append(Paragraph(
    'Trata-se de habeas corpus impetrado pela Defensoria Pública do Estado de São Paulo '
    'em favor de CARLOS EDUARDO MENDONÇA, contra acórdão do TJSP que manteve a prisão '
    'preventiva do paciente. O paciente foi preso em flagrante em 15/03/2024, sendo-lhe '
    'imputada a prática do art. 33, caput, da Lei n. 11.343/2006 (Lei de Drogas). '
    'Laudo IC-SP-2024-045821 comprovou 87,3g de cloridrato de cocaína apreendidos.', estilo_corpo))

elementos.append(Paragraph('<b>VOTO</b>', estilo_negrito))
elementos.append(Paragraph(
    'A presente impetração merece prosperar. O decreto prisional se limita a afirmar que '
    'a gravidade do crime justifica a custódia, sem indicar elementos concretos. '
    'A jurisprudência desta Corte é consolidada no sentido de que a gravidade abstrata '
    'do delito não constitui fundamentação idônea. Precedentes: HC 789.234-SP, HC 756.901-RJ.', estilo_corpo))

elementos.append(Paragraph('<b>ACÓRDÃO</b>', estilo_negrito))
elementos.append(Paragraph(
    'Por unanimidade, CONCEDER a ordem de habeas corpus para revogar a prisão preventiva, '
    'com imposição de medidas cautelares (art. 319 do CPP): comparecimento mensal em juízo '
    'e proibição de ausentar-se da comarca sem autorização.', estilo_corpo))

doc.build(elementos)
print(f"✅ PDF criado em: {CAMINHO_PDF}")
print(f"📄 Tamanho: {os.path.getsize(CAMINHO_PDF)/1024:.1f} KB")

## 📊 Etapa 2: Comparação — Extração Simples vs. Docling

Antes de usar o Docling, vamos ver como a extração com `pdfplumber`
processa o documento. Isso nos dará uma linha de base para comparação.

**⏱️ Tempo estimado: 5 minutos**


In [ ]:
import pdfplumber

print("=" * 60)
print("EXTRAÇÃO COM PDFPLUMBER (baseline)")
print("=" * 60)

texto_pdfplumber = ''
with pdfplumber.open(CAMINHO_PDF) as pdf:
    for i, pagina in enumerate(pdf.pages):
        texto_pagina = pagina.extract_text() or ''
        texto_pdfplumber += texto_pagina
        print(f"📄 Página {i+1}: {len(texto_pagina)} chars")

print(f"\n📊 Total extraído: {len(texto_pdfplumber)} chars")
print("\n--- AMOSTRA (primeiros 600 chars) ---")
print(texto_pdfplumber[:600])

### 🚀 Extração com Docling

O Docling analisa o layout antes de extrair o texto, gerando Markdown estruturado.


In [ ]:
from docling.document_converter import DocumentConverter

print("=" * 60)
print("EXTRAÇÃO COM DOCLING")
print("=" * 60)
print("⏳ Convertendo... (10-20s na primeira execução)")

converter = DocumentConverter()
resultado = converter.convert(CAMINHO_PDF)
doc_docling = resultado.document
markdown_docling = doc_docling.export_to_markdown()

print(f"\n✅ Conversão concluída!")
print(f"📄 Páginas: {len(doc_docling.pages)}")
print(f"📊 Caracteres: {len(markdown_docling)}")
print("\n--- MARKDOWN DOCLING (primeiros 1000 chars) ---")
print(markdown_docling[:1000])

In [ ]:
print("=" * 60)
print("COMPARAÇÃO QUANTITATIVA")
print("=" * 60)

metricas = [
    ("Total de caracteres", len(texto_pdfplumber), len(markdown_docling)),
    ("Tem headings (##)", "##" in texto_pdfplumber, "##" in markdown_docling),
    ("Menciona EMENTA", "EMENTA" in texto_pdfplumber, "EMENTA" in markdown_docling),
    ("Tem negrito (**)", "**" in texto_pdfplumber, "**" in markdown_docling),
]

print(f"\n{'Métrica':<40} {'pdfplumber':<15} {'Docling':<15}")
print("-" * 70)
for nome, vp, vd in metricas:
    vantagem = "← Docling ✅" if str(vd) not in ["False", "0"] and str(vp) in ["False", "0"] else ""
    print(f"{nome:<40} {str(vp):<15} {str(vd):<15} {vantagem}")

print("\n💡 O Docling preserva a estrutura semântica do documento.")

## 🔍 Etapa 3: Explorando o DoclingDocument

O `DoclingDocument` é o objeto central do Docling. Ele representa o documento
de forma hierárquica e permite acesso a todos os elementos detectados.

**⏱️ Tempo estimado: 8 minutos**


In [ ]:
print("=" * 60)
print("ESTRUTURA DO DOCLINGDOCUMENT")
print("=" * 60)

print(f"\n📑 Propriedades do documento:")
print(f"   • Páginas: {len(doc_docling.pages)}")
print(f"   • Textos: {len(doc_docling.texts)}")
print(f"   • Tabelas: {len(doc_docling.tables)}")
print(f"   • Figuras: {len(doc_docling.pictures)}")

from docling_core.types.doc import TextItem, TableItem, PictureItem, SectionHeaderItem

contagem = {'SectionHeader': 0, 'Paragraph': 0, 'Table': 0, 'Outros': 0}
for item, nivel in doc_docling.iterate_items():
    if isinstance(item, SectionHeaderItem):
        contagem['SectionHeader'] += 1
    elif isinstance(item, TableItem):
        contagem['Table'] += 1
    elif isinstance(item, TextItem):
        contagem['Paragraph'] += 1
    else:
        contagem['Outros'] += 1

print("\n📋 Elementos por tipo:")
for tipo, qtd in contagem.items():
    print(f"   • {tipo}: {qtd}")

print("\n📌 Seções detectadas:")
for item, nivel in doc_docling.iterate_items():
    if isinstance(item, SectionHeaderItem):
        indent = '  ' * nivel
        print(f"   {indent}[Nível {nivel+1}] {item.text[:80]}")

In [ ]:
doc_json = doc_docling.export_to_dict()
print("Chaves do JSON exportado:")
for chave, valor in doc_json.items():
    tipo = type(valor).__name__
    tamanho = f"({len(valor)} items)" if isinstance(valor, (list, dict)) else f"= {str(valor)[:40]}"
    print(f"  • {chave}: {tipo} {tamanho}")

caminho_json = '/tmp/acordao_docling.json'
with open(caminho_json, 'w', encoding='utf-8') as f:
    json.dump(doc_json, f, ensure_ascii=False, indent=2)
print(f"\n✅ JSON salvo em: {caminho_json}")
print(f"📊 Tamanho: {os.path.getsize(caminho_json)/1024:.1f} KB")

## ✂️ Etapa 4: Chunking com HybridChunker

O `HybridChunker` nativo do Docling divide o documento em chunks que
respeitam a estrutura semântica — sem cortar no meio de artigos ou seções.

**⏱️ Tempo estimado: 8 minutos**


In [ ]:
from docling.chunking import HybridChunker

chunker = HybridChunker(
    tokenizer='BAAI/bge-m3',
    max_tokens=400,
    merge_peers=True,
)

print("⏳ Gerando chunks...")
chunks = list(chunker.chunk(doc_docling))

print(f"\n✅ Total de chunks: {len(chunks)}")
tamanhos = [len(c.text) for c in chunks]
print(f"📊 Distribuição (chars): min={min(tamanhos)} | max={max(tamanhos)} | média={sum(tamanhos)/len(tamanhos):.0f}")

print("\n" + "=" * 60)
print("PRIMEIROS 3 CHUNKS:")
print("=" * 60)
for i, chunk in enumerate(chunks[:3]):
    print(f"\n─── Chunk {i+1} ({len(chunk.text)} chars) ───")
    print(chunk.text[:350])
    if hasattr(chunk, 'meta') and hasattr(chunk.meta, 'headings') and chunk.meta.headings:
        print(f"Hierarquia: {" > ".join(chunk.meta.headings)}")

In [ ]:
def extrair_metadados_juridicos(chunk_texto, metadados_base=None):
    # Extrai metadados juridicos do texto do chunk
    meta = metadados_base.copy() if metadados_base else {}
    padrao_cnj = r'\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}'
    numeros = re.findall(padrao_cnj, chunk_texto)
    if numeros:
        meta['numero_processo'] = numeros[0]
    artigos = re.findall(r'art\.\s*(\d+[\xba\xb0]?)', chunk_texto, re.IGNORECASE)
    if artigos:
        meta['artigos_mencionados'] = list(set(artigos[:5]))
    leis = re.findall(r'Lei\s+(?:n[\xba\xb0]?\.?\s*)?(\d+[/.]\d+)', chunk_texto)
    if leis:
        meta['leis_citadas'] = list(set(leis[:5]))
    for sigla in ['STF', 'STJ', 'TJ', 'TRF']:
        if sigla in chunk_texto:
            meta['tribunal'] = sigla
            break
    kws = ['habeas corpus', 'prisao preventiva', 'trafico', 'hediondo']
    meta['palavras_chave'] = [k for k in kws if k.lower() in chunk_texto.lower()]
    meta['data_ingestao'] = datetime.now().isoformat()
    meta['num_chars'] = len(chunk_texto)
    return meta

metadados_base = {
    'tipo_documento': 'acordao',
    'tribunal': 'STJ',
    'arquivo': 'acordao_stj_exemplo.pdf',
    'versao_docling': '2.x'
}

chunks_enriquecidos = []
for i, chunk in enumerate(chunks):
    meta = extrair_metadados_juridicos(chunk.text, metadados_base.copy())
    meta['chunk_index'] = i
    chunks_enriquecidos.append({'texto': chunk.text, 'metadados': meta})

print(f"✅ {len(chunks_enriquecidos)} chunks enriquecidos")
print("\nExemplo do chunk 0:")
print(json.dumps(chunks_enriquecidos[0]['metadados'], ensure_ascii=False, indent=2))

## ✅ Checkpoint — Verificação dos Resultados

Execute a célula abaixo para confirmar que todos os objetivos do Lab 1 foram atingidos.


In [ ]:
print("=" * 60)
print("CHECKPOINT — LAB 1")
print("=" * 60)
erros = []

# 1. PDF criado?
if not Path(CAMINHO_PDF).exists():
    erros.append('❌ PDF de exemplo não foi criado')
else:
    print("✅ PDF de exemplo criado com sucesso")

# 2. Docling converteu?
if len(markdown_docling) < 200:
    erros.append('❌ Docling extraiu menos texto que o esperado')
else:
    print(f"✅ Docling extraiu {len(markdown_docling)} caracteres")

# 3. Chunks gerados?
if len(chunks) < 2:
    erros.append('❌ Menos de 2 chunks gerados')
else:
    print(f"✅ {len(chunks)} chunks gerados pelo HybridChunker")

# 4. Metadados enriquecidos?
if 'tribunal' not in chunks_enriquecidos[0]['metadados']:
    erros.append('❌ Metadados jurídicos não extraídos')
else:
    print(f"✅ Metadados extraídos (tribunal: {chunks_enriquecidos[0]['metadados'].get('tribunal', 'N/A')})")

if erros:
    for e in erros: print(e)
    print("\n⚠️ Revise os passos anteriores")
else:
    print("\n🎉 TODOS OS OBJETIVOS ATINGIDOS! Avance para o Lab 2.")

## 🏋️ Exercício — Explore por Conta Própria

**Desafio:** Modifique o chunker para `max_tokens=200` e compare:
1. Quantos chunks são gerados?
2. A hierarquia de headings é preservada?
3. Algum chunk ficou com menos de 50 chars?


In [ ]:
# 🏋️ EXERCÍCIO: Experimente diferentes configurações do HybridChunker
# Modifique os valores e analise as diferenças!

chunker_pequeno = HybridChunker(
    tokenizer='BAAI/bge-m3',
    max_tokens=200,  # Mude para 150, 300, 500...
    merge_peers=True
)
chunks_pequenos = list(chunker_pequeno.chunk(doc_docling))
print(f"Chunks com max_tokens=200: {len(chunks_pequenos)}")
tamanhos_p = [len(c.text) for c in chunks_pequenos]
print(f"Tamanho médio: {sum(tamanhos_p)/len(tamanhos_p):.0f} chars")
print(f"Chunks < 50 chars: {sum(1 for t in tamanhos_p if t < 50)}")
print("\n💭 Qual configuração parece melhor para textos jurídicos?")

In [ ]:
Path('/tmp/aula5_resultados').mkdir(exist_ok=True)

# Salvar chunks enriquecidos
with open('/tmp/aula5_resultados/chunks_lab1.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_enriquecidos, f, ensure_ascii=False, indent=2)

# Salvar markdown extraído
with open('/tmp/aula5_resultados/acordao_markdown.md', 'w', encoding='utf-8') as f:
    f.write(markdown_docling)

print("✅ Resultados salvos em /tmp/aula5_resultados/")
print(f"   • chunks_lab1.json ({len(chunks_enriquecidos)} chunks)")
print(f"   • acordao_markdown.md ({len(markdown_docling)} chars)")
print("\n🔜 Estes arquivos serão usados nos próximos laboratórios.")

## 📚 Resumo — O que aprendemos no Lab 1

Neste laboratório você aprendeu a:

1. **Instalar e configurar** o Docling 2.x no Google Colab
2. **Comparar** extração simples (pdfplumber) vs. estruturada (Docling)
3. **Explorar** o objeto `DoclingDocument` (headings, parágrafos, tabelas)
4. **Exportar** o documento em Markdown e JSON estruturado
5. **Usar o HybridChunker** para divisão inteligente respeitando a hierarquia
6. **Enriquecer chunks** com metadados jurídicos automáticos

### Próximos labs:
- **Lab 2:** PDFs escaneados com OCR (laudos periciais)
- **Lab 3:** Extração de tabelas e formulários jurídicos
- **Lab 4:** Pipeline de ingestão em escala
- **Lab 5:** Pipeline RAG completo com OpenSearch


## 📖 Referências (ABNT)

AUER, Peter et al. **Docling Technical Report**. arXiv:2408.09869, 2024. IBM Research Europe.
Disponível em: <https://arxiv.org/abs/2408.09869>. Acesso em: abr. 2026.

IBM RESEARCH. **Docling Documentation**. Disponível em: <https://ds4sd.github.io/docling/>.
Acesso em: abr. 2026.

CHEN, J. et al. **BGE M3-Embedding**. arXiv:2309.07597, 2024.
